### Read data from Kafka

In [0]:
# Kafka Ingestion using Micro-Batch Approach
# Reads data from Kafka in batch mode to simulate streaming due to serverless environment limitations.

df_batch = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "pkc-921jm.us-east-2.aws.confluent.cloud:9092")
    .option("subscribe", "smartgear_orders")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="I73YUVFYOFS7YLR4" password="cflti2Qkly5Sp2l3t0DzbmDugbhd26qndOLWUGai8WZ9MFavs1oaVorl9CRlJkqw";'
    )
    .load()
)

df_batch.show(5)

+----+--------------------+----------------+---------+------+--------------------+-------------+
| key|               value|           topic|partition|offset|           timestamp|timestampType|
+----+--------------------+----------------+---------+------+--------------------+-------------+
|NULL|[7B 22 6F 72 64 6...|smartgear_orders|        5|     0|2026-04-13 14:38:...|            0|
|NULL|[7B 22 6F 72 64 6...|smartgear_orders|        5|     1|2026-04-13 14:38:...|            0|
|NULL|[7B 22 6F 72 64 6...|smartgear_orders|        5|     2|2026-04-13 14:38:...|            0|
|NULL|[7B 22 6F 72 64 6...|smartgear_orders|        5|     3|2026-04-13 14:38:...|            0|
|NULL|[7B 22 6F 72 64 6...|smartgear_orders|        5|     4|2026-04-13 14:38:...|            0|
+----+--------------------+----------------+---------+------+--------------------+-------------+
only showing top 5 rows


### Prepare Kafka data for parsing

In [0]:
# Convert Kafka message value from binary to string for JSON parsing

df_string = df_batch.selectExpr("CAST(value AS STRING)")
df_string.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                   |
+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"order_id": 9540, "timestamp": "2026-04-13 14:38:10.611754", "store_id": 5, "product": "Tablet", "quantity": 5, "price": 292.94, "region": "East"}     |
|{"order_id": 5015, "timestamp": "2026-04-13 14:38:20.844274", "store_id": 2, "product": "Headphones", "quantity": 2, "price": 656.4, "region": "North"} |
|{"order_id": 3429, "timestamp": "2026-04-13 14:38:22.847782", "store_id": 7, "product": "Laptop", "quantity": 5, "price": 970.01, "region": "South"}    |
|{"order_id": 7614, "timestamp": "2026-04-13 14:38:36.975723", "store_

### Parse JSON + Schema enforcement

In [0]:
# Parse JSON string into structured columns using schema

from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("store_id", IntegerType()),
    StructField("product", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price", DoubleType()),
    StructField("region", StringType())
])

parsed_df = (
    df_string
    .select(from_json(col("value"), schema).alias("data"))
    .select("data.*")
)

parsed_df.show(5)

+--------+--------------------+--------+----------+--------+------+------+
|order_id|           timestamp|store_id|   product|quantity| price|region|
+--------+--------------------+--------+----------+--------+------+------+
|    9540|2026-04-13 14:38:...|       5|    Tablet|       5|292.94|  East|
|    5015|2026-04-13 14:38:...|       2|Headphones|       2| 656.4| North|
|    3429|2026-04-13 14:38:...|       7|    Laptop|       5|970.01| South|
|    7614|2026-04-13 14:38:...|       1|Headphones|       3|778.83| South|
|    9573|2026-04-13 14:38:...|      10|     Phone|       1| 808.8| South|
+--------+--------------------+--------+----------+--------+------+------+
only showing top 5 rows


### Adding ingestion_time & source

In [0]:
# Add metadata columns for Bronze layer

from pyspark.sql.functions import current_timestamp, lit

bronze_df = (
    parsed_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source", lit("kafka"))
)

### Write to Bronze layer

In [0]:
# Write processed data to Delta Bronze layer

parsed_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_smartgear_orders")

### Ensuring data is written

In [0]:
# validate bronze
spark.table("bronze_smartgear_orders").show(5)

+--------+--------------------+--------+----------+--------+------+------+
|order_id|           timestamp|store_id|   product|quantity| price|region|
+--------+--------------------+--------+----------+--------+------+------+
|    9540|2026-04-13 14:38:...|       5|    Tablet|       5|292.94|  East|
|    5015|2026-04-13 14:38:...|       2|Headphones|       2| 656.4| North|
|    3429|2026-04-13 14:38:...|       7|    Laptop|       5|970.01| South|
|    7614|2026-04-13 14:38:...|       1|Headphones|       3|778.83| South|
|    9573|2026-04-13 14:38:...|      10|     Phone|       1| 808.8| South|
+--------+--------------------+--------+----------+--------+------+------+
only showing top 5 rows


Until now
Data has:
Been ingested from Kafka
Converted from binary → JSON
Parsed into structured format
Enriched with metadata
Due to Databricks Serverless limitations, data is displayed instead of being persisted